# 🧩 Hierarchical Visual Classifier — RoBERTa (Kaggle T4)

Fine-tunes a **two-level hierarchical visual-template classifier** for the
slides-generator, ported from `train_classifier.py`.

**Differences from the original `.py`:**
- Base model is **RoBERTa** (`roberta-base`) instead of DistilBERT.
- Tuned for a **single T4 GPU** (16 GB) on Kaggle (fp16, batch 16).

**Everything else is faithful to the original training logic:**
- Level 1 = category (5 classes: data_structure, flow_diagram, chart,
  architectural, conceptual); Level 2 = template within each category.
- `none` is **not** a class — it was a sink that hurt precision. Rows labeled
  `none` are skipped at load time; "render no visual" is a confidence-threshold
  decision at inference, not a learned class.
- Focal Loss (γ=2.0) with √-frequency α-weights for class imbalance.
- Cosine LR schedule + warmup, differential per-layer-group learning rates.
- Stratified 70/15/15 split with a balanced test set.
- WordNet synonym augmentation for rare classes (<80 samples).
- Early stopping (patience=3 on `f1_macro`), per-class report + confusion matrix.

---
## 1 · Setup

Upload the verified training JSONL (e.g. `final_balanced_verified.jsonl`) as a
Kaggle **Dataset** and attach it (it lands under `/kaggle/input/...`). Set the
GPU to **T4** in *Settings → Accelerator*.

In [ ]:
# Kaggle usually has transformers/torch preinstalled. Pin compatible versions
# and ensure nltk is present. Safe to re-run.
!pip -q install -U "transformers>=4.40,<5" "scikit-learn>=1.3" nltk 2>/dev/null
import transformers, torch
print("transformers:", transformers.__version__)
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os, json, random, glob
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE.upper())

In [ ]:
# ── Plot aesthetics + output dirs (Save-&-Run-All friendly) ──────────────────
# Everything written under /kaggle/working is persisted as notebook output and
# is downloadable after a committed run. We give matplotlib a modern, non-default
# look (muted palette, soft grid, no top/right spines, high DPI) and a save_fig()
# helper that persists each chart AND renders it inline.
import re
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cycler
import seaborn as sns
import pandas as pd
from IPython.display import display

FIG_DIR = Path("/kaggle/working/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)

PALETTE   = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3",
             "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD"]
ACCENT    = "#2D3142"        # near-black for text
GRID      = "#E7E7EE"
HEAT_CMAP = sns.color_palette("rocket_r", as_cmap=True)

sns.set_theme(style="white", context="notebook")
plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 200,
    "figure.facecolor": "white", "savefig.facecolor": "white", "savefig.bbox": "tight",
    "axes.edgecolor": "#B9B9C6", "axes.linewidth": 1.0,
    "axes.titlesize": 14, "axes.titleweight": "bold", "axes.titlepad": 14,
    "axes.titlecolor": ACCENT, "axes.labelcolor": ACCENT,
    "axes.labelsize": 11, "axes.labelweight": "medium",
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 1.0,
    "xtick.color": ACCENT, "ytick.color": ACCENT,
    "xtick.labelsize": 10, "ytick.labelsize": 10,
    "axes.prop_cycle": cycler(color=PALETTE),
    "font.size": 11, "legend.frameon": False,
    "figure.titleweight": "bold", "figure.titlesize": 16,
})

def _slug(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

def _despine(ax, left=True, bottom=True):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    if not left:   ax.spines["left"].set_visible(False)
    if not bottom: ax.spines["bottom"].set_visible(False)

def save_fig(fig, name):
    """Persist a figure to /kaggle/working/figures (downloadable) and show it."""
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path)
    print(f"  💾 {path}")
    plt.show(); plt.close(fig)

# Populated by train_one_level() during training: level_name -> arrays/history.
RESULTS = {}
print("🎨 Plot theme ready · figures →", FIG_DIR)

## 2 · Configuration

`DATA_PATH` auto-discovers the JSONL under `/kaggle/input`. Override it if needed.

In [ ]:
# ── Auto-locate the training file under /kaggle/input ────────────────────────
def _find_data():
    candidates = glob.glob("/kaggle/input/**/classifier_train*.jsonl", recursive=True)
    if candidates:
        # Prefer the regenerated_cleaned file if present
        for c in candidates:
            if "regenerated_cleaned" in c:
                return c
        return candidates[0]
    # Local fallback (running outside Kaggle)
    local = "classifier_train_regenerated_cleaned.jsonl"
    return local if os.path.exists(local) else ""

DATA_PATH    = _find_data()
OUTPUT_DIR   = "/kaggle/working/visual_classifier_roberta"
MODEL_NAME   = "roberta-base"

# ── Hyperparameters (T4-tuned) ───────────────────────────────────────────────
EPOCHS        = 30      # early stopping usually halts well before this
BATCH_SIZE    = 16      # roberta-base @ max_len 256 fits comfortably on a T4
LEARNING_RATE = 2e-5
WARMUP_STEPS  = 50
MAX_LENGTH    = 256
AUGMENT       = True

# RoBERTa-base has 12 encoder layers — split lower/upper at layer 6 for
# differential learning rates (DistilBERT used 6 layers split at 3).
LOWER_UPPER_SPLIT = 6

assert DATA_PATH, "❌ Could not find classifier_train*.jsonl — set DATA_PATH manually."
print("Data:  ", DATA_PATH)
print("Output:", OUTPUT_DIR)
print("Model: ", MODEL_NAME)

## 3 · Hierarchy definitions

Embedded copy of `slide_gen/core/hierarchy.py` so the notebook is self-contained.

In [ ]:
# NOTE: "none" is intentionally NOT a category. It used to be a sink class that
# absorbed uncertain predictions (it cratered conceptual/chart precision). Any
# "none"-labeled rows are skipped at load time; "render no visual" is a
# confidence-threshold decision at inference, not a learned class.
CATEGORY_HIERARCHY = {
    "data_structure": ["linear_chain", "binary_tree", "general_tree", "stack", "queue", "graph"],
    "flow_diagram":   ["flowchart", "cycle"],
    "chart":          ["bar_chart"],
    "architectural":  ["architecture_diagram"],
    "conceptual":     ["conceptual"],
}

CATEGORY_LIST  = list(CATEGORY_HIERARCHY.keys())
CATEGORY_TO_ID = {c: i for i, c in enumerate(CATEGORY_LIST)}

TEMPLATE_TO_CATEGORY = {}
for _cat, _templates in CATEGORY_HIERARCHY.items():
    for _t in _templates:
        TEMPLATE_TO_CATEGORY.setdefault(_t, _cat)
TEMPLATE_TO_CATEGORY["conceptual"] = "conceptual"

LEVEL2_LABELS = {c: t for c, t in CATEGORY_HIERARCHY.items() if t}
LEVEL2_LABEL_TO_ID = {c: {l: i for i, l in enumerate(ls)} for c, ls in LEVEL2_LABELS.items()}

def get_category(template_id: str) -> str:
    # default "none" = unknown / not a trained category → skipped at load time
    return TEMPLATE_TO_CATEGORY.get(template_id, "none")

# Quick peek at the dataset's label distribution
_counts = Counter()
with open(DATA_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            _counts[json.loads(line).get("label", "none")] += 1
print("Template distribution:")
for k, v in _counts.most_common():
    print(f"  {k:22s}: {v}")
print("\nCategory distribution (trained categories only; 'none' shown as dropped):")
_cat = Counter()
for k, v in _counts.items():
    _cat[get_category(k)] += v
for k, v in _cat.most_common():
    tag = "  (DROPPED)" if k not in CATEGORY_TO_ID else ""
    print(f"  {k:22s}: {v}{tag}")

## 4 · WordNet synonym augmentation (rare classes)

In [ ]:
import nltk
_WORDNET = False
try:
    from nltk.corpus import wordnet
    try:
        wordnet.synsets("test"); _WORDNET = True
    except LookupError:
        nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
        wordnet.synsets("test"); _WORDNET = True
except Exception as e:
    print("⚠️  WordNet unavailable — augmentation disabled:", e)

AUGMENTATION_THRESHOLD = 80
AUGMENTATION_REPLACE_PROB = 0.15

def synonym_replace(text, replace_prob=AUGMENTATION_REPLACE_PROB):
    if not _WORDNET:
        return text
    out = []
    for word in text.split():
        if len(word) < 4 or random.random() > replace_prob:
            out.append(word); continue
        syns = set()
        for syn in wordnet.synsets(word.lower()):
            for lemma in syn.lemmas():
                name = lemma.name().replace("_", " ")
                if name.lower() != word.lower():
                    syns.add(name)
        out.append(random.choice(list(syns)) if syns else word)
    return " ".join(out)

print("WordNet augmentation:", "enabled" if _WORDNET else "disabled")

## 5 · Dataset, Focal Loss, metrics

In [ ]:
from transformers import AutoTokenizer

class HierarchicalDataset(Dataset):
    """Produces labels for Level 1 (category) or a specific Level 2 category."""
    def __init__(self, data_path, tokenizer, level="category", max_length=MAX_LENGTH, augment=True):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.level = level
        self.texts, self.labels = [], []
        self.augmentation_stats = {}

        if level == "category":
            self.label_list = CATEGORY_LIST
            self.label_to_id = CATEGORY_TO_ID
        else:
            if level not in LEVEL2_LABELS:
                raise ValueError(f"Unknown category: {level}")
            self.label_list = LEVEL2_LABELS[level]
            self.label_to_id = LEVEL2_LABEL_TO_ID[level]

        skipped = 0
        with open(data_path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                ex = json.loads(line)
                template_id = ex.get("label", "none")
                if level == "category":
                    cat = get_category(template_id)
                    if cat not in self.label_to_id:
                        skipped += 1  # 'none'/unknown — not a trained category
                        continue
                    self.texts.append(ex["text"]); self.labels.append(self.label_to_id[cat])
                else:
                    if get_category(template_id) != level:
                        continue
                    if template_id not in self.label_to_id:
                        continue
                    self.texts.append(ex["text"]); self.labels.append(self.label_to_id[template_id])

        msg = f"  Loaded {len(self.texts)} examples for level='{level}' ({len(self.label_list)} classes)"
        if level == "category" and skipped:
            msg += f"  [skipped {skipped} 'none'/unknown]"
        print(msg)
        label_counts = Counter(self.labels)
        for lid in range(len(self.label_list)):
            if label_counts.get(lid, 0):
                print(f"    {self.label_list[lid]}: {label_counts[lid]}")

        if augment and _WORDNET:
            added_total = 0
            for lid in range(len(self.label_list)):
                count = label_counts.get(lid, 0)
                if 0 < count < AUGMENTATION_THRESHOLD:
                    idxs = [i for i, l in enumerate(self.labels) if l == lid]
                    for _ in range(AUGMENTATION_THRESHOLD - count):
                        src = random.choice(idxs)
                        self.texts.append(synonym_replace(self.texts[src]))
                        self.labels.append(lid)
                    added = AUGMENTATION_THRESHOLD - count
                    self.augmentation_stats[self.label_list[lid]] = {
                        "original": count, "augmented": added, "final": count + added}
                    added_total += added
                    print(f"    ↳ Augmented {self.label_list[lid]}: {count} → {count + added}")
            if added_total:
                print(f"  \U0001f4c8 Augmented samples added: {added_total} (total: {len(self.texts)})")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_length,
                             truncation=True, padding="max_length", return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


class FocalLoss(torch.nn.Module):
    """-α(1-p)^γ log(p)  — down-weights easy examples, focuses on hard ones."""
    def __init__(self, alpha, gamma=2.0):
        super().__init__(); self.alpha = alpha; self.gamma = gamma

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=-1)
        oh = F.one_hot(targets, num_classes=logits.size(-1)).float()
        pt = torch.clamp((probs * oh).sum(dim=-1), min=1e-8)
        alpha_t = self.alpha.to(logits.device)[targets]
        loss = -alpha_t * (1 - pt) ** self.gamma * torch.log(pt)
        return loss.mean()

In [ ]:
from sklearn.model_selection import train_test_split as sklearn_split
from sklearn.utils import resample as sklearn_resample
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, balanced_accuracy_score,
                             matthews_corrcoef, cohen_kappa_score)

def make_balanced_test_set(indices, labels):
    labels = np.array(labels); unique = np.unique(labels)
    counts = Counter(labels.tolist()); min_count = min(counts.values())
    if min_count < 1:
        return list(indices), 0
    balanced = []
    for cls in unique:
        cls_idx = np.array(indices)[labels == cls]
        balanced.extend(sklearn_resample(cls_idx, n_samples=min_count,
                                         replace=False, random_state=SEED).tolist())
    return balanced, min_count

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    top3 = np.argsort(logits, axis=-1)[:, -3:]
    top3_acc = np.mean([l in t for l, t in zip(labels, top3)])
    return {
        "accuracy": accuracy_score(labels, preds),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "top3_accuracy": top3_acc,
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(labels, preds),
    }

## 6 · Single-level trainer (RoBERTa)

Differential learning rates are mapped onto RoBERTa's module names
(`roberta.embeddings`, `roberta.encoder.layer[...]`, `classifier`). RoBERTa's
classification head (`RobertaClassificationHead`) replaces DistilBERT's
`pre_classifier` + `classifier` pair.

In [ ]:
from transformers import (AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, EarlyStoppingCallback)

def train_one_level(dataset, label_list, output_dir, model_name,
                    epochs, batch_size, learning_rate, warmup_steps, level_name):
    num_labels = len(label_list)
    out_path = Path(output_dir); out_path.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 70)
    print(f"TRAINING: {level_name} ({num_labels} classes)  | epochs: {epochs} | metric: f1_macro")
    print("=" * 70)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels)
    # RoBERTa dropout knobs (analogue of DistilBERT's seq_classif_dropout/dropout)
    model.config.hidden_dropout_prob = 0.2
    model.config.attention_probs_dropout_prob = 0.1
    if hasattr(model.config, "classifier_dropout"):
        model.config.classifier_dropout = 0.3

    all_indices = list(range(len(dataset)))
    all_labels = np.array(dataset.labels)
    if len(dataset) < 10:
        print(f"  ⚠️  Too few samples ({len(dataset)}) — skipping {level_name}."); return None

    label_counts_split = Counter(all_labels.tolist())
    min_class_count = min(label_counts_split.values())
    balanced_test_info = None

    if min_class_count < 3:
        print(f"  ⚠️  A class has only {min_class_count} samples — random split")
        n = len(dataset); tr = int(0.7 * n); va = int(0.15 * n)
        train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
            dataset, [tr, va, n - tr - va])
    else:
        train_idx, temp_idx = sklearn_split(all_indices, test_size=0.30,
                                            stratify=all_labels, random_state=SEED)
        val_idx, test_idx = sklearn_split(temp_idx, test_size=0.50,
                                          stratify=all_labels[temp_idx], random_state=SEED)
        bal_test_idx, min_per_class = make_balanced_test_set(test_idx, all_labels[test_idx])
        train_dataset = Subset(dataset, train_idx)
        val_dataset = Subset(dataset, val_idx)
        test_dataset = Subset(dataset, bal_test_idx)
        balanced_test_info = {"min_per_class": min_per_class, "total": len(bal_test_idx)}

    print(f"  Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
    if balanced_test_info:
        print(f"  Test balanced: {balanced_test_info['min_per_class']} samples/class")

    # ── Focal Loss α weights (√-frequency, normalized, clamped) ──────────────
    label_counts = Counter(dataset.labels); total = len(dataset.labels)
    alpha = torch.ones(num_labels)
    for lid, c in label_counts.items():
        if c > 0:
            alpha[lid] = np.sqrt(total / c)
    alpha = torch.clamp(alpha / alpha.mean(), max=10.0)
    focal_loss_fn = FocalLoss(alpha=alpha, gamma=2.0)
    print("  ⚖️  Focal α:", {label_list[i]: round(float(alpha[i]), 2)
                              for i in range(num_labels) if label_counts.get(i, 0) > 0})

    # ── Differential learning rates (RoBERTa module names) ──────────────────
    enc_layers = model.roberta.encoder.layer
    param_groups = [
        {"params": list(model.roberta.embeddings.parameters()),         "lr": learning_rate * 0.1},
        {"params": list(enc_layers[:LOWER_UPPER_SPLIT].parameters()),    "lr": learning_rate * 0.5},
        {"params": list(enc_layers[LOWER_UPPER_SPLIT:].parameters()),    "lr": learning_rate * 1.0},
        {"params": list(model.classifier.parameters()),                 "lr": learning_rate * 2.0},
    ]

    class FocalTrainer(Trainer):
        def __init__(self, *a, _param_groups=None, **kw):
            self._param_groups = _param_groups; super().__init__(*a, **kw)
        def compute_loss(self, model, inputs, return_outputs=False, **kw):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss = focal_loss_fn(outputs.logits, labels)
            return (loss, outputs) if return_outputs else loss
        def create_optimizer(self):
            if self._param_groups is not None:
                self.optimizer = torch.optim.AdamW(self._param_groups, weight_decay=0.01)
            else:
                return super().create_optimizer()

    args = TrainingArguments(
        output_dir=str(out_path), num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=batch_size,
        warmup_steps=warmup_steps, learning_rate=learning_rate, weight_decay=0.01,
        label_smoothing_factor=0.1, lr_scheduler_type="cosine", max_grad_norm=1.0,
        # Log train loss every EPOCH (not every 50 steps) so small levels — e.g.
        # flow_diagram, whose epoch is < 50 steps — still get a train-loss point
        # per epoch instead of nothing for the first few epochs.
        logging_strategy="epoch", eval_strategy="epoch", save_strategy="epoch",
        save_total_limit=1, save_only_model=True, load_best_model_at_end=True,
        metric_for_best_model="f1_macro", greater_is_better=True, report_to="none",
        fp16=torch.cuda.is_available(), dataloader_pin_memory=torch.cuda.is_available(),
    )

    trainer = FocalTrainer(
        model=model, args=args, train_dataset=train_dataset, eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        _param_groups=param_groups,
    )

    print("  Training (early stopping patience=3)...")
    trainer.train()
    trainer.save_model(str(out_path)); tokenizer.save_pretrained(str(out_path))
    with open(out_path / "label_config.json", "w") as f:
        json.dump({"label_list": label_list, "level_name": level_name}, f, indent=2)

    # ── Test-set evaluation ──────────────────────────────────────────────────
    print("  📊 TEST SET (balanced):")
    res = trainer.predict(test_dataset)
    logits = res.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()   # row-wise softmax
    preds = np.argmax(logits, axis=-1)
    true = np.array([test_dataset[i]["labels"].item() for i in range(len(test_dataset))])
    present = sorted(set(true.tolist()) | set(preds.tolist()))
    names = [label_list[i] for i in present]
    report = classification_report(true, preds, labels=present, target_names=names, zero_division=0)
    print(report)
    with open(out_path / "test_evaluation.txt", "w") as f:
        f.write(f"TEST SET EVALUATION ({level_name})\n")
        f.write(f"Accuracy:    {accuracy_score(true, preds):.4f}\n")
        f.write(f"F1 Macro:    {f1_score(true, preds, average='macro', zero_division=0):.4f}\n")
        f.write(f"F1 Weighted: {f1_score(true, preds, average='weighted', zero_division=0):.4f}\n\n")
        f.write(report)

    # ── Stash everything the visualization section needs (Save-&-Run-All) ────
    RESULTS[level_name] = {
        "true": true,
        "preds": preds,
        "probs": probs,
        "label_list": list(label_list),
        "present": present,
        "names": names,
        "log_history": list(trainer.state.log_history),
        "augmentation_stats": dict(getattr(dataset, "augmentation_stats", {})),
    }
    return trainer

## 7 · Full hierarchical training

Phase 1 trains the Level-1 category classifier; Phase 2 trains a Level-2
template classifier for each category that has >1 template and enough samples.

In [ ]:
MIN_SAMPLES_FOR_LEVEL2 = 10

# Per-level epoch caps, read off the train/val-loss curves to stop before
# overfitting. (load_best_model_at_end + early stopping remain a backstop, but
# these caps avoid wasting epochs and keep the saved curves clean.)
EPOCHS_PER_LEVEL = {
    "category":       5,   # Level 1 (category classifier)
    "data_structure": 8,   # Level 2
    "flow_diagram":   4,   # Level 2 (flowchart + cycle) — best ~epoch 4, overfits after
}

def train_full(data_path, output_dir, model_name=MODEL_NAME, epochs=EPOCHS,
               batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
               warmup_steps=WARMUP_STEPS, augment=AUGMENT):
    out_base = Path(output_dir); out_base.mkdir(parents=True, exist_ok=True)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    print("\n" + "█" * 70 + "\nPHASE 1: LEVEL 1 — CATEGORY CLASSIFIER\n" + "█" * 70)
    l1 = HierarchicalDataset(data_path, tokenizer, level="category", augment=augment)
    l1_epochs = EPOCHS_PER_LEVEL.get("category", epochs)
    train_one_level(l1, CATEGORY_LIST, str(out_base / "level1"), model_name,
                    l1_epochs, batch_size, learning_rate, warmup_steps, "Level 1 (Category)")

    print("\n\n" + "█" * 70 + "\nPHASE 2: LEVEL 2 — PER-CATEGORY TEMPLATE CLASSIFIERS\n" + "█" * 70)
    trained, skipped = [], []
    for category, templates in LEVEL2_LABELS.items():
        if len(templates) <= 1:
            print(f"\n  ⏭️  Skip '{category}' — single template"); skipped.append(category); continue
        l2 = HierarchicalDataset(data_path, tokenizer, level=category, augment=augment)
        if len(l2) < MIN_SAMPLES_FOR_LEVEL2:
            print(f"\n  ⏭️  Skip '{category}' — only {len(l2)} samples"); skipped.append(category); continue
        l2_epochs = EPOCHS_PER_LEVEL.get(category, epochs)
        train_one_level(l2, templates, str(out_base / "level2" / category), model_name,
                        l2_epochs, batch_size, learning_rate, warmup_steps, f"Level 2 ({category})")
        trained.append(category)

    with open(out_base / "hierarchy_config.json", "w") as f:
        json.dump({"category_list": CATEGORY_LIST,
                   "category_hierarchy": CATEGORY_HIERARCHY,
                   "trained_level2": trained, "skipped_level2": skipped}, f, indent=2)
    print("\n" + "=" * 70)
    print("TRAINING COMPLETE")
    print(f"  Level 1:       {out_base / 'level1'}")
    print(f"  Level 2:       {', '.join(trained) or '(none)'}")
    print(f"  Skipped:       {', '.join(skipped) or '(none)'}")
    print("=" * 70)

train_full(DATA_PATH, OUTPUT_DIR)

## 8 · Evaluation metrics & visualizations

All charts are computed from the **balanced test set** captured during training
(`RESULTS`) and saved to `/kaggle/working/figures/` so they survive a
**Save & Run All** commit and are downloadable from the *Output* tab.

- **Headline table** — accuracy, balanced accuracy, macro/weighted F1, MCC,
  Cohen's κ, top-3 accuracy, macro ROC-AUC / PR-AUC, and **ECE** (calibration).
- **Training dynamics** — loss and validation F1/accuracy per epoch.
- **Confusion matrix** — counts annotated, colored by per-row recall.
- **Per-class P/R/F1** — with support counts, sorted by F1.
- **ROC & PR curves** — one-vs-rest, per-class AUC.
- **Calibration** — reliability diagram + confidence distribution (correct vs wrong).

In [ ]:
# ── Headline metrics per classifier level ────────────────────────────────────
# Beyond accuracy we report metrics that matter under class imbalance and for a
# downstream compound-AI system: balanced accuracy, macro-F1, MCC, Cohen's κ,
# top-3 accuracy, macro ROC-AUC / PR-AUC, and ECE (calibration error).
from sklearn.metrics import roc_auc_score, average_precision_score


def expected_calibration_error(probs, true, n_bins=15):
    """Weighted gap between confidence and accuracy across confidence bins."""
    conf = probs.max(axis=1); pred = probs.argmax(axis=1)
    acc = (pred == true).astype(float); n = len(true); bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            ece += (m.sum() / n) * abs(acc[m].mean() - conf[m].mean())
    return float(ece)


def extended_metrics(r):
    true, preds, probs = r["true"], r["preds"], r["probs"]
    out = {
        "accuracy":      accuracy_score(true, preds),
        "balanced_acc":  balanced_accuracy_score(true, preds),
        "f1_macro":      f1_score(true, preds, average="macro", zero_division=0),
        "f1_weighted":   f1_score(true, preds, average="weighted", zero_division=0),
        "mcc":           matthews_corrcoef(true, preds),
        "cohen_kappa":   cohen_kappa_score(true, preds),
        "ece":           expected_calibration_error(probs, true),
        "n_classes":     len(r["present"]),
        "n_test":        len(true),
    }
    out["top3_acc"] = (np.mean([t in row for t, row in
                                zip(true, np.argsort(probs, axis=1)[:, -3:])])
                       if probs.shape[1] >= 3 else np.nan)
    try:
        # Binary-safe one-hot ([N, n_cls] always — label_binarize collapses to a
        # single column for 2-class levels like flow_diagram, which broke AUC).
        n_cls = probs.shape[1]
        y_bin = np.zeros((len(true), n_cls), dtype=int)
        y_bin[np.arange(len(true)), np.asarray(true)] = 1
        cols = [c for c in range(n_cls) if y_bin[:, c].sum() > 0]
        if len(cols) >= 2:
            out["roc_auc_macro"] = roc_auc_score(y_bin[:, cols], probs[:, cols], average="macro")
            out["pr_auc_macro"]  = average_precision_score(y_bin[:, cols], probs[:, cols], average="macro")
    except Exception:
        out["roc_auc_macro"] = np.nan; out["pr_auc_macro"] = np.nan
    return out


if not RESULTS:
    print("⚠️  RESULTS is empty — run the training cell (Section 7) first.")
else:
    summary = pd.DataFrame({lvl: extended_metrics(r) for lvl, r in RESULTS.items()}).T
    summary.to_csv(FIG_DIR / "summary_metrics.csv")
    print("💾", FIG_DIR / "summary_metrics.csv")

    # Heatmap of the bounded [0,1] metrics (ECE shown separately — lower is better).
    bounded = ["accuracy", "balanced_acc", "f1_macro", "f1_weighted",
               "top3_acc", "mcc", "cohen_kappa", "roc_auc_macro", "pr_auc_macro"]
    cols = [c for c in bounded if c in summary.columns]
    H = summary[cols].astype(float)
    fig, ax = plt.subplots(figsize=(1.15 * len(cols) + 2.5, 0.75 * len(H) + 2))
    sns.heatmap(H, annot=True, fmt=".3f", cmap="mako", vmin=0, vmax=1,
                linewidths=2.5, linecolor="white", cbar=False,
                annot_kws={"weight": "bold", "size": 10}, ax=ax)
    ax.set_title("Headline metrics by classifier level  ·  ECE (lower=better) in table below")
    ax.set_xticklabels([c.replace("_", " ") for c in cols], rotation=38, ha="right")
    ax.set_yticklabels(H.index, rotation=0)
    save_fig(fig, "00_summary_metrics")

    display(summary.round(4))

In [ ]:
# ── Per-level diagnostic plots ───────────────────────────────────────────────
# Training curves · confusion matrix · per-class P/R/F1 · ROC & PR · calibration.
from sklearn.metrics import (precision_recall_fscore_support, roc_curve, auc,
                             precision_recall_curve)


def _onehot(true, n_cls):
    """Robust [N, n_cls] one-hot (avoids label_binarize's binary 1-column collapse)."""
    y = np.zeros((len(true), n_cls), dtype=int)
    y[np.arange(len(true)), np.asarray(true)] = 1
    return y


def plot_training_curves(level, r):
    hist = pd.DataFrame(r["log_history"])
    if hist.empty:
        return
    tr = hist[hist["loss"].notna()] if "loss" in hist else pd.DataFrame()
    ev = hist[hist["eval_loss"].notna()] if "eval_loss" in hist else pd.DataFrame()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    ax = axes[0]
    if not tr.empty and "epoch" in tr:
        ax.plot(tr["epoch"], tr["loss"], color=PALETTE[0], lw=2.2,
                marker="o", ms=4, label="train loss")
    if not ev.empty and "epoch" in ev:
        ax.plot(ev["epoch"], ev["eval_loss"], color=PALETTE[3], lw=2.2,
                marker="o", ms=5, label="val loss")
    ax.set_title("Loss"); ax.set_xlabel("epoch"); ax.set_ylabel("loss")
    ax.legend(); _despine(ax)
    ax = axes[1]
    for col, color, lbl in [("eval_f1_macro", PALETTE[2], "val F1 macro"),
                            ("eval_accuracy", PALETTE[4], "val accuracy")]:
        if not ev.empty and col in ev:
            ax.plot(ev["epoch"], ev[col], color=color, lw=2.2, marker="o", ms=5, label=lbl)
            b = ev[col].idxmax()
            ax.scatter(ev.loc[b, "epoch"], ev.loc[b, col], color=color, s=150,
                       zorder=5, edgecolor="white", linewidth=1.6)
    ax.set_title("Validation metrics"); ax.set_xlabel("epoch")
    ax.set_ylim(0, 1.02); ax.legend(); _despine(ax)
    fig.suptitle(f"Training dynamics — {level}", y=1.03)
    save_fig(fig, f"01_training_{_slug(level)}")


def plot_confusion(level, r):
    true, preds, present, names = r["true"], r["preds"], r["present"], r["names"]
    cm = confusion_matrix(true, preds, labels=present)
    cmn = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    side = 0.85 * len(names)
    fig, ax = plt.subplots(figsize=(side + 3, side + 2.5))
    sns.heatmap(cmn, annot=cm, fmt="d", cmap=HEAT_CMAP, vmin=0, vmax=1,
                xticklabels=names, yticklabels=names, square=True,
                linewidths=1.5, linecolor="white",
                cbar_kws={"label": "row-normalized rate (recall)", "shrink": 0.78},
                annot_kws={"size": 10, "weight": "bold"}, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion matrix — {level}\ncell = count · color = recall")
    plt.setp(ax.get_xticklabels(), rotation=40, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    save_fig(fig, f"02_confusion_{_slug(level)}")


def plot_per_class(level, r):
    true, preds, present, names = r["true"], r["preds"], r["present"], r["names"]
    p, rec, f1, sup = precision_recall_fscore_support(
        true, preds, labels=present, zero_division=0)
    order = np.argsort(f1)                       # worst at bottom, best on top
    nm = [names[i] for i in order]
    p, rec, f1, sup = p[order], rec[order], f1[order], sup[order]
    y = np.arange(len(nm)); h = 0.26
    fig, ax = plt.subplots(figsize=(9, 0.55 * len(nm) + 2.5))
    for i, (vals, color, lbl) in enumerate(
            [(p, PALETTE[0], "precision"), (rec, PALETTE[1], "recall"), (f1, PALETTE[2], "F1")]):
        ax.barh(y + (1 - i) * h, vals, height=h, color=color, label=lbl, edgecolor="white")
    for yi, s in zip(y, sup):
        ax.text(1.012, yi, f"n={int(s)}", va="center", ha="left", fontsize=9, color="#8C8C8C")
    ax.set_yticks(y); ax.set_yticklabels(nm); ax.set_xlim(0, 1.0)
    ax.set_xlabel("score"); ax.set_title(f"Per-class precision / recall / F1 — {level}")
    ax.legend(ncol=3, loc="lower right"); _despine(ax, left=False)
    save_fig(fig, f"03_perclass_{_slug(level)}")


def plot_roc_pr(level, r):
    true, probs, label_list = r["true"], r["probs"], r["label_list"]
    n_cls = probs.shape[1]
    classes = [c for c in range(n_cls) if (true == c).sum() > 0]
    if len(classes) < 2:
        print(f"  ⏭️  {level}: <2 populated classes — skipping ROC/PR"); return
    y_bin = _onehot(true, n_cls)                  # always [N, n_cls] (binary-safe)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for i, c in enumerate(classes):
        color = PALETTE[i % len(PALETTE)]; nm = label_list[c]
        fpr, tpr, _ = roc_curve(y_bin[:, c], probs[:, c]); a = auc(fpr, tpr)
        axes[0].plot(fpr, tpr, color=color, lw=2, label=f"{nm} ({a:.2f})")
        pr, rc, _ = precision_recall_curve(y_bin[:, c], probs[:, c]); ap = auc(rc, pr)
        axes[1].plot(rc, pr, color=color, lw=2, label=f"{nm} ({ap:.2f})")
    axes[0].plot([0, 1], [0, 1], "--", color="#B9B9C6")
    axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")
    axes[0].set_title("ROC (one-vs-rest)")
    axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
    axes[1].set_title("Precision–Recall (one-vs-rest)")
    for ax in axes:
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
        ax.legend(fontsize=8, title="class (AUC)"); _despine(ax)
    fig.suptitle(f"ROC & PR — {level}", y=1.03)
    save_fig(fig, f"04_roc_pr_{_slug(level)}")


def plot_calibration(level, r):
    probs, true, preds = r["probs"], r["true"], r["preds"]
    conf = probs.max(axis=1); correct = (preds == true).astype(float)
    n_bins = 12; bins = np.linspace(0, 1, n_bins + 1)
    acc = np.full(n_bins, np.nan); avgc = np.full(n_bins, np.nan)
    for b in range(n_bins):
        m = (conf > bins[b]) & (conf <= bins[b + 1])
        if m.sum() > 0:
            acc[b] = correct[m].mean(); avgc[b] = conf[m].mean()
    ece = expected_calibration_error(probs, true)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    ax = axes[0]
    ax.plot([0, 1], [0, 1], "--", color="#B9B9C6", label="perfect")
    v = ~np.isnan(acc)
    ax.plot(avgc[v], acc[v], "-o", color=PALETTE[0], lw=2.2, ms=7, label="model")
    ax.bar((bins[:-1] + bins[1:]) / 2, np.nan_to_num(acc), width=0.9 / n_bins,
           color=PALETTE[0], alpha=0.15)
    ax.set_xlabel("confidence"); ax.set_ylabel("accuracy")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title(f"Reliability diagram (ECE = {ece:.3f})"); ax.legend(); _despine(ax)
    ax = axes[1]
    ax.hist(conf[correct == 1], bins=15, range=(0, 1), color=PALETTE[2],
            alpha=0.75, label="correct", edgecolor="white")
    ax.hist(conf[correct == 0], bins=15, range=(0, 1), color=PALETTE[3],
            alpha=0.75, label="incorrect", edgecolor="white")
    ax.set_xlabel("max softmax confidence"); ax.set_ylabel("count")
    ax.set_title("Confidence distribution"); ax.legend(); _despine(ax)
    fig.suptitle(f"Calibration — {level}", y=1.03)
    save_fig(fig, f"05_calibration_{_slug(level)}")


if not RESULTS:
    print("⚠️  RESULTS is empty — run the training cell first.")
for _lvl, _r in RESULTS.items():
    print(f"\n📈 {_lvl}")
    plot_training_curves(_lvl, _r)
    plot_confusion(_lvl, _r)
    plot_per_class(_lvl, _r)
    plot_roc_pr(_lvl, _r)
    plot_calibration(_lvl, _r)

## 9 · Quick inference sanity check

Loads the trained Level-1 model and predicts the category of a sample chunk.

In [ ]:
from transformers import AutoModelForSequenceClassification as _AMSC

_l1_dir = Path(OUTPUT_DIR) / "level1"
_tok = AutoTokenizer.from_pretrained(str(_l1_dir))
_model = _AMSC.from_pretrained(str(_l1_dir)).to(DEVICE).eval()
_labels = json.load(open(_l1_dir / "label_config.json"))["label_list"]

def predict_category(text):
    enc = _tok(text, max_length=MAX_LENGTH, truncation=True,
               padding="max_length", return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(_model(**enc).logits, dim=-1)[0].cpu().numpy()
    order = probs.argsort()[::-1]
    return [(_labels[i], round(float(probs[i]), 3)) for i in order[:3]]

for t in [
    "A stack follows LIFO. Push adds to the top, pop removes from the top.",
    "The transformer encoder uses multi-head self-attention and feed-forward layers.",
    "First check if the input is valid. If yes, parse it. Otherwise log an error and return.",
]:
    print(t[:60], "->", predict_category(t))

## 10 · Outputs

All trained models and figures are already saved under `/kaggle/working/`
(session storage). After a **Save & Run All** commit, download them individually
from the *Output* tab — no zipping needed.

In [ ]:
# Everything is already written to /kaggle/working (session storage) — no zipping.
# After a Save & Run All commit these are downloadable from the 'Output' tab.
from pathlib import Path

print("📂 Models →", OUTPUT_DIR)
for p in sorted(Path(OUTPUT_DIR).rglob("*")):
    if p.is_file():
        print("   ", p.relative_to("/kaggle/working"))

print("\n🖼️  Figures →", FIG_DIR)
for p in sorted(FIG_DIR.glob("*")):
    print("   ", p.relative_to("/kaggle/working"))

print("\n✅ Download any of these from the Kaggle 'Output' tab.")